In [ ]:
# ================================
# Milestone 2 - Data Preparation
# Azure Demand Forecasting Project
# ================================

import pandas as pd
import os

# ------------------------------
# 1. Paths
# ------------------------------
DATA_DIR = "../data"
USAGE_PATH = os.path.join(DATA_DIR, "azure_usage.csv")
EXT_PATH   = os.path.join(DATA_DIR, "external_factors.csv")
OUTPUT_PATH = os.path.join(DATA_DIR, "cleaned_merged.csv")

# ------------------------------
# 2. Load datasets
# ------------------------------
usage = pd.read_csv(USAGE_PATH, parse_dates=["date"])
ext = pd.read_csv(EXT_PATH, parse_dates=["date"])

print("✅ Usage shape:", usage.shape, " External shape:", ext.shape)

# ------------------------------
# 3. Merge on date
# ------------------------------
df = pd.merge(usage, ext, on="date", how="left")

# Standardize column names
if "cpu_used" in df.columns and "usage_cpu" not in df.columns:
    df = df.rename(columns={"cpu_used": "usage_cpu"})
if "storage_used" in df.columns and "usage_storage" not in df.columns:
    df = df.rename(columns={"storage_used": "usage_storage"})

# ------------------------------
# 4. Feature Engineering
# ------------------------------

# CPU Utilization (per resource or global if no resource_id)
if "resource_id" in df.columns:
    df["cpu_utilization"] = df.groupby("resource_id")["usage_cpu"].transform(lambda x: x / x.max())
    df["storage_efficiency"] = df.groupby("resource_id")["usage_storage"].transform(lambda x: x / x.max())
else:
    df["cpu_utilization"] = df["usage_cpu"] / df["usage_cpu"].max()
    df["storage_efficiency"] = df["usage_storage"] / df["usage_storage"].max()

# Time-based features
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["quarter"] = df["date"].dt.quarter

# Lag features
df = df.sort_values("date")
df["cpu_lag_1"] = df["usage_cpu"].shift(1)
df["cpu_lag_7"] = df["usage_cpu"].shift(7)

# Rolling averages
df["cpu_rolling_7"] = df["usage_cpu"].rolling(7, min_periods=1).mean()
df["cpu_rolling_30"] = df["usage_cpu"].rolling(30, min_periods=1).mean()

# Fill missing values
df = df.fillna(method="ffill").fillna(method="bfill").fillna(0)

# ------------------------------
# 5. Save processed dataset
# ------------------------------
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved processed dataset: {OUTPUT_PATH}")
df.head(10)


✅ Usage shape: (1080, 6)  External shape: (90, 4)
✅ Saved processed dataset: ../data\cleaned_merged.csv


C:\Users\kbhuk\AppData\Local\Temp\ipykernel_5508\1546453749.py:64: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill").fillna(0)


,date,region,resource_type,usage_cpu,usage_storage,users_active,economic_index,cloud_market_demand,holiday,cpu_utilization,storage_efficiency,month,day_of_week,is_weekend,quarter,cpu_lag_1,cpu_lag_7,cpu_rolling_7,cpu_rolling_30
0,2023-01-01,East US,VM,88,1959,470,104.97,0.99,1,0.888889,0.981955,1,6,1,1,88.0,88.0,88.000000,88.000000
11,2023-01-01,Southeast Asia,Container,77,1199,470,104.97,0.99,1,0.777778,0.601003,1,6,1,1,88.0,88.0,82.500000,82.500000
10,2023-01-01,Southeast Asia,Storage,76,1582,369,104.97,0.99,1,0.767677,0.792982,1,6,1,1,77.0,88.0,80.333333,80.333333
9,2023-01-01,Southeast Asia,VM,93,1356,248,104.97,0.99,1,0.939394,0.679699,1,6,1,1,76.0,88.0,83.500000,83.500000
7,2023-01-01,North Europe,Storage,51,1715,476,104.97,0.99,1,0.515152,0.859649,1,6,1,1,93.0,88.0,77.000000,77.000000
6,2023-01-01,North Europe,VM,73,1937,493,104.97,0.99,1,0.737374,0.970927,1,6,1,1,51.0,88.0,76.333333,76.333333
8,2023-01-01,North Europe,Container,82,959,221,104.97,0.99,1,0.828283,0.480702,1,6,1,1,73.0,88.0,77.142857,77.142857
4,2023-01-01,West US,Storage,85,1371,351,104.97,0.99,1,0.858586,0.687218,1,6,1,1,82.0,88.0,76.714286,78.125000
3,2023-01-01,West US,VM,60,1982,287,104.97,0.99,1,0.606061,0.993484,1,6,1,1,85.0,77.0,74.285714,76.111111
2,2023-01-01,East US,Container,70,621,414,104.97,0.99,1,0.707071,0.311278,1,6,1,1,60.0,76.0,73.428571,75.500000


: 